# Data Processing

## Set Config

In [14]:
# ライブラリのインポート
import os
import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import seaborn as sns

In [15]:
# フォルダ設定
data_dir = '../data/'
output_dir = '../output/'

# date
date = '20260316_emergency_first24h'

# 特徴量tableのファイル名
features_file_name = f'{date}_features.csv'
df_development_file_name = f'{date}_df_development_raw.csv'

In [16]:
# 保存するファイル名の指定
df_development_train_file_name = f'{date}_df_development_train.csv'
df_development_test_file_name = f'{date}_df_development_test.csv'

## Load Data

In [17]:
# データの読み込み
df_development = pl.read_csv(os.path.join(data_dir + df_development_file_name))

In [18]:
# record数と症例数を確認
print('Number of records (model development cohort):', len(df_development))
print('Number of icu_stay_id (model development cohort):', len(df_development['icu_stay_id'].unique()))
print('Average number of records per icu_stay_id (model development cohort):', len(df_development) / len(df_development['icu_stay_id'].unique()))

Number of records (model development cohort): 27307
Number of icu_stay_id (model development cohort): 27307
Average number of records per icu_stay_id (model development cohort): 1.0


In [19]:
# データの最初を見てみる。読み込めているか確認
df_development.head()

icu_stay_id,time_window_index,start_time,end_time,time_zero,out_time,label_delirium_indication,ph,pao2,pvo2,paco2,pvco2,bg_lactate,bg_glucose,gcs_e,gcs_v,gcs_m,body_weight,bt50,hr10,hr50,hr90,hr_sd,rr10,rr50,rr90,rr_sd,sbp10,sbp50,sbp90,sbp_sd,mbp10,mbp50,mbp90,mbp_sd,dbp10,dbp50,…,time_since_last_control_ventilation,mv_duration,rass_score,respiration_24hours,coagulation_24hours,liver_24hours,cardiovascular_24hours,cns_24hours,renal_24hours,sofa_24hours,input_amount,active_ingredient_propofol,active_ingredient_noradrenaline,active_ingredient_fentanyl,active_ingredient_dexmedetomidine,active_ingredient_dopamine,active_ingredient_midazolam,active_ingredient_dobutamine,active_ingredient_landiolol,active_ingredient_vasopressin,active_ingredient_remifentanil,active_ingredient_thiamylal,active_ingredient_adrenaline,active_ingredient_vecuronium,active_ingredient_rocuronium,window_total_output,transfusion_rbc,transfusion_platelet,transfusion_plasma,transfusion_albumin,fio2_ordered_is_current_value,peep_ordered_is_current_value,pao2_is_current_value,standardized_urinary_output,cumsum_in,cumsum_out,cumsum_balance
i64,i64,str,str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,f64,f64,f64,f64
28023193256164,0,"""2200-01-01 21:03:00 UTC""","""2200-01-02 21:03:00 UTC""","""2200-01-01 21:03:00 UTC""","""2200-01-10 10:00:00 UTC""",0,7.4425,91.0,53.3,28.8,28.0,3.2,331.5,4.0,4.0,6.0,52.1,36.5,59.0,68.0,93.0,13.826292,14.0,18.0,22.0,4.765907,90.0,113.0,146.0,21.212549,69.0,83.0,97.0,12.914838,52.0,65.0,…,0,38,0.0,1.0,0.0,1.0,1.0,2.0,2.0,7.0,489.896358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1129.998,0.0,0.0,0.0,0.0,0,0,1,27.2315,489.896358,1129.998,-640.101642
41011001070839,0,"""2200-01-01 23:30:00 UTC""","""2200-01-02 23:30:00 UTC""","""2200-01-01 23:30:00 UTC""","""2200-01-08 13:37:00 UTC""",0,7.405,112.0,null,37.0,null,0.9,120.0,1.5,1.0,2.5,61.4,37.400002,48.0,62.0,67.0,7.301701,14.0,20.0,24.0,3.961408,101.0,127.0,147.0,18.085077,60.0,76.0,89.0,10.870757,42.0,53.0,…,null,null,-4.0,0.0,2.0,0.0,1.0,0.0,0.0,3.0,84.0,1386.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1770.0,256.666667,0.0,246.666667,0.0,0,0,1,32.0728,84.0,1770.0,-1686.0
48157154283148,0,"""2200-01-01 12:37:00 UTC""","""2200-01-02 12:37:00 UTC""","""2200-01-01 12:37:00 UTC""","""2200-01-07 12:59:00 UTC""",0,null,null,null,null,null,null,null,4.0,5.0,6.0,43.0,36.6,78.0,93.0,111.0,12.409638,14.0,20.0,26.0,4.359502,117.2,128.0,140.4,9.36647,85.2,88.0,110.2,11.721665,66.4,74.0,…,null,null,0.0,null,0.0,0.0,0.0,0.0,3.0,3.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1136.248,0.0,0.0,0.0,0.0,0,0,0,24.0014,23.0,1136.248,-1113.248
57017826690967,0,"""2200-01-01 13:53:00 UTC""","""2200-01-02 13:53:00 UTC""","""2200-01-01 13:53:00 UTC""","""2200-01-18 11:30:00 UTC""",1,7.447,84.85,null,36.4,null,1.01,152.0,1.0,1.0,1.0,null,36.2,44.0,57.0,71.0,10.688538,20.0,20.0,20.0,2.484409,111.0,129.0,147.0,18.374649,69.0,83.0,100.0,14.326949,47.8,60.0,…,0,1371,-4.0,4.0,0.0,0.0,3.0,0.0,4.0,11.0,0.0,5020.833333,2.18115,1.249833,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,621.0,469.999,0.0,0.0,0.0,0.0,1,1,1,11.5803,0.0,469.999,-469.999
84835012925988,0,"""2200-01-01 13:59:00 UTC""","""2200-01-02 13:59:00 UTC""","""2200-01-01 13:59:00 UTC""","""2200-01-06 10:00:00 UTC""",0,null,null,null,null,null,null,null,1.0,1.0,2.0,81.0,36.8,54.0,60.0,87.0,13.004632,12.0,14.0,21.0,4.0726,65.2,95.0,121.4,21.701118,47.6,60.5,77.7,13.582946,31.5,46.0,…,null,null,-3.5,null,0.0,2.0,1.0,4.0,1.0,8.0,0.666667,0.0,0.0,0.0,4.95,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,596.666,0.0,0.0,0.0,0.0,0,0,0,9.2944,0.666667,596.666,-595.999333


In [20]:
### df_development
# レコードの数を計算
records_count = df_development.height

# 再挿管になった症例数を計算
number_of_delirium_records = len(df_development.filter(pl.col('label_delirium_indication') == 1))

# 上記で計算した症例数を表示
print('==== Model Development Cohort ====')
print(f'Sample size is: {records_count}')
print(f'Percentage of delirium: {number_of_delirium_records/records_count*100:.2f}')

==== Model Development Cohort ====
Sample size is: 27307
Percentage of delirium: 24.09


## Train Test Split

In [21]:
# model development cohortをtrain dataとtest dataに分ける関数
def split_on_patients(df: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    unique_ids = df['icu_stay_id'].unique().to_numpy()

    # train_test_splitを使用して症例ベースでtrainとtestに分割
    train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=710)

    # データフレームを分割
    train_df = df.filter(pl.col('icu_stay_id').is_in(train_ids))
    test_df = df.filter(pl.col('icu_stay_id').is_in(test_ids))

    return train_df, test_df

In [22]:
# trainとtestに分割
df_development_train, df_development_test = split_on_patients(df_development)

In [23]:
# 症例数の確認
len(df_development_train['icu_stay_id'].unique()), len(df_development_test['icu_stay_id'].unique())

(21845, 5462)

In [24]:
### df_development train
# レコードの数を計算
records_count = df_development_train.height

# せん妄になった症例数を計算
number_of_delirium_records = len(df_development_train.filter(pl.col('label_delirium_indication') == 1))

# 上記で計算した症例数を表示
print('==== Model Development Train Cohort ====')
print(f'Sample size is: {records_count}')
print(f'Percentage of delirium: {number_of_delirium_records/records_count*100:.2f}')

==== Model Development Train Cohort ====
Sample size is: 21845
Percentage of delirium: 24.04


In [25]:
### df_development test
# レコードの数を計算
records_count = df_development_test.height

# 再挿管になった症例数を計算
number_of_delirium_records = len(df_development_test.filter(pl.col('label_delirium_indication') == 1))

# 上記で計算した症例数を表示
print('==== Model Development Cohort ====')
print(f'Sample size is: {records_count}')
print(f'Percentage delirium: {number_of_delirium_records/records_count*100:.2f}')

==== Model Development Cohort ====
Sample size is: 5462
Percentage delirium: 24.28


## Save as CSV

In [26]:
# csvに保存
df_development_train.write_csv(os.path.join(data_dir, df_development_train_file_name))
df_development_test.write_csv(os.path.join(data_dir, df_development_test_file_name))